In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam


In [ ]:

# 1. Load all training CSVs (normal only)
train_files = glob.glob("SynCAN/train_*.csv")
train_dfs = [pd.read_csv(f) for f in train_files]
train_data = pd.concat(train_dfs, ignore_index=True)

# 2. Load all test CSVs (normal + attack)
test_files = glob.glob("SynCAN/test_*.csv")
test_dfs = [pd.read_csv(f) for f in test_files]
test_data = pd.concat(test_dfs, ignore_index=True)

# 3. Combine train + test
data = pd.concat([train_data, test_data], ignore_index=True)

# 4. Features and labels
# SynCAN has columns like: ID, DLC, Data0..Data7, Label
X = data.drop(columns=["Label"])
y = data["Label"]

# 5. Balance dataset by undersampling normal traffic
normal = data[data["Label"] == 0]
attack = data[data["Label"] == 1]


In [ ]:

# Match counts
normal_downsampled = resample(normal,
                              replace=False,
                              n_samples=len(attack),
                              random_state=42)

balanced = pd.concat([normal_downsampled, attack])

X_bal = balanced.drop(columns=["Label"])
y_bal = balanced["Label"]

# 6. Train/test split
X_train, X_val, y_train, y_val = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
)


In [ ]:

# 7. Simple feedforward NN (like INDRA notebook style)
model = Sequential([
    Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer=Adam(learning_rate=1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy"])


In [ ]:

# 8. Train
history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=10,
                    batch_size=256)


In [ ]:

# 9. Evaluate
loss, acc = model.evaluate(X_val, y_val)
print(f"Validation accuracy: {acc:.3f}")